# Task T4 — Whisper-Tiny Fine-Tuning on Swahili FLEURS (Run E3)

This notebook runs the fine-tuning of `openai/whisper-tiny` on the Kenyan Swahili (`sw_ke`) training split using Google Colab's free T4 GPU.

### Protocol Compliance:
- **Train Set:** `data/splits/train.tsv` (1,902 clips, ~8.0 h)
- **Validation Set:** `data/splits/validation.tsv` (210 clips, ~0.8 h) for checkpoint selection
- **Test Set:** `data/splits/test.tsv` (252 clips) held out strictly until final evaluation
- **Base Checkpoint Revision:** Pinned commit `169d4a4341b33bc18d8881c4b69c2e104e1cc0af` (same as baseline E1)

## 1. Check GPU Availability
Ensure you are running on a T4 GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
!nvidia-smi

## 2. Clone Repository & Pull Cleaned Audio via Git LFS

In [ ]:
!git clone https://github.com/BeyzaESER22/swahili-asr-finetuning.git
%cd swahili-asr-finetuning

# Download the ~2.1 GB of verified clean WAV audio
!git lfs pull --include="data/cleaned/audio/*"

## 3. Install Dependencies

In [ ]:
!pip install -r requirements-models.txt

## 4. Smoke Test (2 Samples Trial)
Runs 2 samples through the complete pipeline (feature extraction, training loop, evaluation, checkpoint saving) in ~20 seconds to confirm everything works.

In [ ]:
!python src/train.py --smoke-test

## 5. Full Whisper-Tiny Fine-Tuning on T4 GPU
Trains on the 1,902 train clips with FP16 precision, selecting the best model based on validation WER.

In [ ]:
!python src/train.py \
  --fp16 \
  --epochs 3 \
  --batch-size 8 \
  --grad-accum 2 \
  --learning-rate 1e-4 \
  --warmup-steps 50 \
  --output-dir results/checkpoints/E3

## 6. Evaluate E3 on Frozen Test Set & Score Benchmark
Evaluates the best fine-tuned model checkpoint on the identical 252 frozen test clips and updates `results/metrics.csv`.

In [ ]:
# Run inference on the 252 test clips
!python src/run_baseline.py --run E3

# Compute WER and CER across all three models (E1, E2, E3)
!python src/score_predictions.py --predictions \
  results/predictions/E1.tsv \
  results/predictions/E2.tsv \
  results/predictions/E3.tsv

## 7. Display Final Benchmark Results

In [ ]:
import pandas as pd
df = pd.read_csv('results/metrics.csv')
display(df)